In [10]:
! pip install nltk pyodbc sqlalchemy 

In [11]:
import pandas as pd
import pyodbc
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer


In [12]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\rudre\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [13]:
def fetch_data_from_sql():
    conn_str =(
        "Driver={SQL Server};"
        r"Server=LAPTOP-T5KE6GTD\SQLEXPRESS;"
        "Database=PortfolioProject_MarketingAnalytics;"
        "Trusted_Connection=yes;"
    )

    conn = pyodbc.connect(conn_str)

    query = "SELECT ReviewID, CustomerID, ProductID, ReviewDate, Rating, ReviewText FROM [dbo].[customer_reviews]"

    df= pd.read_sql(query,conn)
    conn.close()

    return df

In [14]:
customer_reviews_df = fetch_data_from_sql()

sia = SentimentIntensityAnalyzer()

C:\Users\rudre\AppData\Local\Temp\ipykernel_22640\3370425898.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df= pd.read_sql(query,conn)


In [20]:
#Define a function to calculate sentiment scores using VADER

def calculate_sentiment(review):
    sentiments=sia.polarity_scores(review)
     # Return the compound score, which is a normalized score between -1 (most negative) and 1 (most positive)
    return sentiments['compound']

In [25]:
# Define a function to categorize sentiment using both the sentiment score and the review rating

def categorize_sentiment(score, rating):
    if score >= 0.5:
        if rating >=4:
            return 'positive'
        elif rating == 3:
            return 'mixed positive'
        else:
            return 'mixed negative'
    elif score < -0.5:
        if rating <= 2:
            return 'negative' 
        elif rating ==3 :
            return 'mixed negative'
        else:
            return' positive'
    else:
        if rating >=4:
            return 'positive'
        elif rating <=2:
            return 'negative'
        else:
            return 'nutral'


In [17]:
def sentiment_bucket(score):
    if score >= 0.5:
        return '0.5 to 1.0'  # Strongly positive sentiment
    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49'  # Mildly positive sentiment
    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'  # Mildly negative sentiment
    else:
        return '-1.0 to -0.5'

In [21]:
customer_reviews_df['SentimentScore'] = customer_reviews_df['ReviewText'].apply(calculate_sentiment)


In [26]:
# Apply sentiment categorization using both text and rating
customer_reviews_df['SentimentCategory'] = customer_reviews_df.apply(
    lambda row: categorize_sentiment(row['SentimentScore'], row['Rating']), axis=1)


In [28]:
customer_reviews_df['SentimentBucket'] = customer_reviews_df['SentimentScore'].apply(sentiment_bucket)


In [29]:
print(customer_reviews_df.head())

   ReviewID  CustomerID  ProductID  ReviewDate  Rating  \
0         1          77         18  2023-12-23       3   
1         2          80         19  2024-12-25       5   
2         3          50         13  2025-01-26       4   
3         4          78         15  2025-04-21       3   
4         5          64          2  2023-07-16       3   

                                 ReviewText  SentimentScore SentimentCategory  \
0   Average  experience,  nothing  special.         -0.3089            nutral   
1            The  quality  is    top-notch.          0.0000          positive   
2   Five  stars  for  the  quick  delivery.          0.0000          positive   
3  Good  quality,  but  could  be  cheaper.          0.2382            nutral   
4   Average  experience,  nothing  special.         -0.3089            nutral   

  SentimentBucket  
0    -0.49 to 0.0  
1     0.0 to 0.49  
2     0.0 to 0.49  
3     0.0 to 0.49  
4    -0.49 to 0.0  


In [33]:
customer_reviews_df.to_csv('fact_customer_reviews_with_sentiment.csv', index=False)